# 6. Export per-video annotation bundles

Merge the model-generated event annotations (notebook 5) with their YOLO spatial coordinates into one JSON bundle per video under `data/processed/<video-name>/meva_export/`:

- `<video-name>.activities.json` — generated events, their descriptions, the checklist `actions` the model saw (plus `other_actions` outside the checklist), the object IDs taking part, and the frame/second range of the window they came from.
- `<video-name>.geom.json` — per-frame bounding boxes (`x_min`/`y_min`/`x_max`/`y_max`), label, confidence, and grid position, kept only for objects referenced by an event.
- `<video-name>.types.json` — one record per object ID: its YOLO labels, the model's free-text physical descriptions, detection counts, min/average/max confidence, observed grid positions, and first/last seen times.
- `<video-name>.metadata.json` — source path, duration, frame count, fps, resolution, FOURCC, camera ID, scene, and the capture-time range parsed from the video name.

**On the name "MEVA-style".** The four files mirror MEVA's file layout (an `activities` / `geom` / `types` triple per video, plus a metadata file here), but the contents are *not* MEVA KPF. Real MEVA annotations are YAML records such as `{ act: { act2: {person_opens_facility_door: 1.0}, id2: ..., timespan: [{tsr0: [start, end]}] } }` and `{ geom: { g0: "245 617 420 728", id1: 12, ts0: 2949 } }`; this notebook writes plain JSON with its own field names (`event_name`, `actors[].object_id`, `bounding_box_pixels`, `source_status`). MEVA tooling and KPF parsers cannot read these files — they exist to be loaded by notebook 7 and the annotation explorer. The output directory is still called `meva_export/` for compatibility with existing runs.

These are also **model-generated annotations**, not ground truth: event labels keep the local model's vocabulary, object types are raw YOLO class names (no mapping to MEVA's `cset3` vocabulary), and every activity carries `source_status: "generated"`.

**Rerun behaviour** is controlled by `OVERWRITE`:

- `OVERWRITE = False` (default): a video is exported only if any of its four export files is missing, or if its `window_event_annotations.json` / `yolo_spatial_coordinates.json` are newer than the export (notebook 5 was rerun).
- `OVERWRITE = True`: every video is exported again.

When run from `00_run_annotation_pipeline.ipynb`, the runner's `OVERWRITE` takes precedence.

In [1]:
import hashlib
import json
import os
import re
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import cv2

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebooks").is_dir() and (parent / "annotation_pipeline").is_dir()
)
RAW_VIDEOS = PROJECT_ROOT / "data" / "raw" / "videos"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# True: regenerate the MEVA-style export for every video.
# False: only export videos whose export files are missing or older than their annotations.
OVERWRITE = True  # Pilot: only re-annotated videos get a new export, so notebook 07 reloads only those.
OVERWRITE = os.environ.get("PIPELINE_OVERWRITE", str(OVERWRITE)).strip().lower() in {"1", "true", "yes"}
EXPORT_SUFFIXES = ("activities", "geom", "types", "metadata")


def export_is_current(video_name, export_dir, input_paths):
    """True when all export files exist and are at least as new as every input file."""
    output_paths = [export_dir / f"{video_name}.{suffix}.json" for suffix in EXPORT_SUFFIXES]
    if not all(path.is_file() for path in output_paths):
        return False
    newest_input = max(path.stat().st_mtime for path in input_paths)
    return min(path.stat().st_mtime for path in output_paths) >= newest_input

def parse_video_name(video_name):
    """Extract the capture interval, scene, and camera ID embedded in MEVA names."""
    match = re.fullmatch(
        r"(?P<date>\d{4}-\d{2}-\d{2})\.(?P<start>\d{2}-\d{2}-\d{2})\."
        r"(?P<end>\d{2}-\d{2}-\d{2})\.(?P<scene>[^.]+)\.(?P<camera>[^.]+)",
        video_name,
    )
    if not match:
        return {"video_name": video_name}

    values = match.groupdict()
    capture_start = datetime.strptime(
        f"{values['date']} {values['start']}", "%Y-%m-%d %H-%M-%S"
    )
    capture_end = datetime.strptime(
        f"{values['date']} {values['end']}", "%Y-%m-%d %H-%M-%S"
    )
    return {
        "video_name": video_name,
        "scene": values["scene"],
        "camera_id": values["camera"],
        "capture_start_local": capture_start.isoformat(),
        "capture_end_local": capture_end.isoformat(),
        "capture_time_zone": "unknown",
    }


def find_source_video(video_name):
    """Support both original AVI names and VideoMeta's optional .r13 suffix."""
    candidates = [RAW_VIDEOS / f"{video_name}.avi", RAW_VIDEOS / f"{video_name}.r13.avi"]
    return next((path for path in candidates if path.is_file()), None)


event_paths = sorted(PROCESSED_DIR.glob("*/window_event_annotations.json"))
if not event_paths:
    raise FileNotFoundError(f"No event annotations found in {PROCESSED_DIR}")
print("=" * 72, flush=True)
print("Stage 6: exporting per-video annotation bundles", flush=True)
print(f"OVERWRITE={OVERWRITE}: {len(event_paths)} annotated videos found.", flush=True)
print("=" * 72, flush=True)
exported_count = skipped_count = 0

for index, event_path in enumerate(event_paths, start=1):
    video_name = event_path.parent.name
    print(f"[{index}/{len(event_paths)}] Exporting {video_name}", flush=True)
    spatial_path = event_path.with_name("yolo_spatial_coordinates.json")
    if not spatial_path.is_file():
        print(f"Skipping {video_name}: missing {spatial_path.name}")
        continue

    export_dir = event_path.parent / "meva_export"
    if not OVERWRITE and export_is_current(video_name, export_dir, [event_path, spatial_path]):
        print(f"Skipping {video_name}: export is up to date")
        skipped_count += 1
        continue

    window_annotations = json.loads(event_path.read_text(encoding="utf-8"))
    spatial_windows = json.loads(spatial_path.read_text(encoding="utf-8"))
    export_dir.mkdir(exist_ok=True)

    tracked_object_ids = {
        str(detection["object_id"])
        for item in spatial_windows
        for detection in item.get("detections", [])
    }
    dropped_actor_ids = set()
    dropped_event_count = 0
    activities = []
    event_object_ids = set()
    object_descriptions = defaultdict(lambda: defaultdict(set))
    activity_id = 0
    for window in window_annotations:
        start_frame = min(
            (detection["frame_index"] for item in spatial_windows
             if item["start_seconds"] == window["start_seconds"]
             for detection in item["detections"]),
            default=None,
        )
        end_frame = max(
            (detection["frame_index"] for item in spatial_windows
             if item["start_seconds"] == window["start_seconds"]
             for detection in item["detections"]),
            default=start_frame,
        )
        if start_frame is None:
            continue
        for event in window.get("events", []):
            for object_item in event.get("involved_objects", []):
                if object_item.get("id") is not None and object_item.get("physical_details"):
                    object_descriptions[str(object_item["id"])][object_item.get("label", "unknown")].add(
                        object_item["physical_details"]
                    )
            mentioned_ids = {
                str(item["id"]) for item in event.get("involved_objects", []) if item.get("id") is not None
            }
            # The model can cite ids the tracker never produced; those have no types/geometry rows.
            actor_ids = sorted(mentioned_ids & tracked_object_ids)
            dropped_actor_ids |= mentioned_ids - tracked_object_ids
            if mentioned_ids and not actor_ids:
                dropped_event_count += 1
                continue
            event_object_ids.update(actor_ids)
            activities.append(
                {
                    "event_id": hashlib.sha256(
                        f"{video_name}:{activity_id}".encode("utf-8")
                    ).hexdigest(),
                    "event_name": event.get("event_name"),
                    "description": event.get("description"),
                    "actions": list(event.get("actions") or []),
                    "other_actions": list(event.get("other_actions") or []),
                    "actors": [{"object_id": object_id} for object_id in actor_ids],
                    "start_frame": start_frame,
                    "end_frame": end_frame,
                    "start_seconds": window["start_seconds"],
                    "end_seconds": window["end_seconds"],
                    "source_status": "generated",
                }
            )
            activity_id += 1

    if dropped_actor_ids or dropped_event_count:
        print(
            f"  {video_name}: dropped untracked actor ids {sorted(dropped_actor_ids, key=int)}"
            f" and {dropped_event_count} event(s) left with no tracked actors"
        )

    detections_by_object = defaultdict(list)
    seen_geometry = set()
    geometries = []
    geometry_id = 0
    for window in spatial_windows:
        for detection in window.get("detections", []):
            object_id = str(detection["object_id"])
            if object_id not in event_object_ids:
                continue
            box = [round(float(value)) for value in detection["box_xyxy"]]
            geometry_key = (object_id, detection["frame_index"], tuple(box))
            if geometry_key in seen_geometry:
                continue
            seen_geometry.add(geometry_key)
            detections_by_object[object_id].append(detection)
            geometries.append(
                {
                    "geometry_id": geometry_id,
                    "object_id": object_id,
                    "frame_index": int(detection["frame_index"]),
                    "timestamp_seconds": detection["timestamp_seconds"],
                    "label": detection["label"],
                    "confidence": detection["confidence"],
                    "spatial_position": detection["spatial_position"],
                    "bounding_box_pixels": {
                        "x_min": box[0],
                        "y_min": box[1],
                        "x_max": box[2],
                        "y_max": box[3],
                    },
                }
            )
            geometry_id += 1

    types = []
    for object_id, detections in sorted(detections_by_object.items(), key=lambda item: int(item[0])):
        timestamps = [detection["timestamp_seconds"] for detection in detections]
        detections_by_label = defaultdict(list)
        for detection in detections:
            detections_by_label[detection["label"]].append(detection)
        types.append(
            {
                "object_id": object_id,
                "detection_count": len(detections),
                "first_seen_seconds": min(timestamps),
                "last_seen_seconds": max(timestamps),
                "labels": [
                    {
                        "label": label,
                        "descriptions": sorted(object_descriptions[object_id][label]),
                        "detection_count": len(label_detections),
                        "confidence": {
                            "minimum": round(
                                min(item["confidence"] for item in label_detections), 4
                            ),
                            "average": round(
                                sum(item["confidence"] for item in label_detections)
                                / len(label_detections),
                                4,
                            ),
                            "maximum": round(
                                max(item["confidence"] for item in label_detections), 4
                            ),
                        },
                        "spatial_positions": sorted(
                            {item["spatial_position"] for item in label_detections}
                        ),
                    }
                    for label, label_detections in sorted(detections_by_label.items())
                ],
            }
        )

    metadata = parse_video_name(video_name)
    source_video = find_source_video(video_name)
    metadata.update(
        {
            "video_id": hashlib.sha256(f"video:{video_name}".encode("utf-8")).hexdigest(),
            "annotation_format": "generated annotation bundle (MEVA file layout, custom JSON schema)",
            "event_annotation_path": str(event_path.relative_to(PROJECT_ROOT)),
            "spatial_annotation_path": str(spatial_path.relative_to(PROJECT_ROOT)),
            "source_video_path": str(source_video.relative_to(PROJECT_ROOT)) if source_video else None,
            "source_video_available": source_video is not None,
            "activity_count": len(activities),
            "object_count": len(types),
            "geometry_count": len(geometries),
        }
    )
    if source_video:
        capture = cv2.VideoCapture(str(source_video))
        metadata["duration_seconds"] = round(
            capture.get(cv2.CAP_PROP_FRAME_COUNT) / capture.get(cv2.CAP_PROP_FPS), 3
        )
        metadata["frame_count"] = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
        metadata["fps"] = round(capture.get(cv2.CAP_PROP_FPS), 6)
        metadata["frame_width"] = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        metadata["frame_height"] = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
        metadata["fourcc"] = "".join(
            chr((int(capture.get(cv2.CAP_PROP_FOURCC)) >> 8 * index) & 0xFF)
            for index in range(4)
        )
        capture.release()

    for suffix, payload in {
        "activities": activities,
        "geom": geometries,
        "types": types,
        "metadata": metadata,
    }.items():
        output_path = export_dir / f"{video_name}.{suffix}.json"
        output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    print(
        f"Exported {video_name}: {len(activities)} activities, "
        f"{len(types)} objects, and {len(geometries)} geometry records"
    )
    exported_count += 1

print(f"Done: {exported_count} exported, {skipped_count} already up to date.")

Stage 6: exporting per-video annotation bundles
OVERWRITE=True: 71 annotated videos found.
[1/71] Exporting 2018-03-05.13-15-00.13-20-00.bus.G340
Exported 2018-03-05.13-15-00.13-20-00.bus.G340: 18 activities, 12 objects, and 4870 geometry records
[2/71] Exporting 2018-03-05.13-15-00.13-20-00.bus.G505
Exported 2018-03-05.13-15-00.13-20-00.bus.G505: 12 activities, 11 objects, and 2918 geometry records
[3/71] Exporting 2018-03-05.13-15-00.13-20-00.bus.G506
Exported 2018-03-05.13-15-00.13-20-00.bus.G506: 18 activities, 11 objects, and 5094 geometry records
[4/71] Exporting 2018-03-05.13-15-00.13-20-00.hospital.G341
Exported 2018-03-05.13-15-00.13-20-00.hospital.G341: 14 activities, 8 objects, and 2566 geometry records
[5/71] Exporting 2018-03-05.13-15-00.13-20-00.school.G423
Exported 2018-03-05.13-15-00.13-20-00.school.G423: 1 activities, 1 objects, and 129 geometry records
[6/71] Exporting 2018-03-05.13-15-00.13-20-00.school.G424
Exported 2018-03-05.13-15-00.13-20-00.school.G424: 40 activ